# Argus Phase 1 — USAspending Data Exploration

Explore the tiny NASA FY2024 samples under `data/samples/usaspending/`.

**Goal:** understand JSON shape, keys, and award ↔ transaction ↔ recipient relationships before designing PostgreSQL schema.

Remember:

```text
HIGH RISK ≠ FRAUD
ANOMALY ≠ MISCONDUCT
```

This notebook is for exploration only. Reusable logic belongs in `src/` later.

In [ ]:
from pathlib import Path
import json
import pandas as pd

ROOT = Path('..').resolve()
SAMPLE_DIR = ROOT / 'data' / 'samples' / 'usaspending'

assert SAMPLE_DIR.exists(), SAMPLE_DIR
sorted(p.name for p in SAMPLE_DIR.iterdir())

## 1. Provenance

Confirm when/how the sample was downloaded and what filters were used.

In [ ]:
provenance = json.loads((SAMPLE_DIR / 'provenance.json').read_text())
provenance

## 2. Award search response structure

In [ ]:
awards_payload = json.loads((SAMPLE_DIR / 'sample_awards_response.json').read_text())
print('top-level keys:', sorted(awards_payload.keys()))
results = awards_payload.get('results', [])
print('n_results:', len(results))
if results:
    print('row keys:', sorted(results[0].keys()))

awards_df = pd.DataFrame(results)
awards_df

In [ ]:
# Null rates for award search fields
awards_df.isna().mean().sort_values(ascending=False)

## 3. Award detail (nested recipient / agency / NAICS / PSC)

In [ ]:
detail = json.loads((SAMPLE_DIR / 'sample_award_detail.json').read_text())
print('top-level keys:', sorted(detail.keys()))
print('generated_unique_award_id:', detail.get('generated_unique_award_id'))
print('id:', detail.get('id'))
print('piid:', detail.get('piid'))
print('recipient:', detail.get('recipient'))
print('awarding_agency keys:', sorted((detail.get('awarding_agency') or {}).keys()))

## 4. Transactions

Inspect both search-style transactions and award-linked transactions.

In [ ]:
tx_search = json.loads((SAMPLE_DIR / 'sample_transactions_response.json').read_text())
tx_award = json.loads((SAMPLE_DIR / 'sample_transactions_by_award.json').read_text())

tx_search_df = pd.DataFrame(tx_search.get('results', []))
tx_award_df = pd.DataFrame(tx_award.get('results', []))

print('search tx keys:', list(tx_search_df.columns))
print('award tx keys:', list(tx_award_df.columns))
display(tx_search_df.head())
display(tx_award_df.head())

## 5. Conceptual join: award → transactions → vendor

Manually verify that sample transactions belong to the detailed award.

In [ ]:
award_key = detail.get('generated_unique_award_id')
award_piid = detail.get('piid')
recipient = (detail.get('recipient') or {})

print('Award natural key:', award_key)
print('PIID:', award_piid)
print('Recipient name:', recipient.get('recipient_name') or recipient.get('name'))
print('Recipient UEI:', recipient.get('uei') or recipient.get('recipient_uei'))

linked = tx_search_df[tx_search_df.get('generated_internal_id') == award_key] if 'generated_internal_id' in tx_search_df.columns else tx_search_df
print('transactions linked by generated_internal_id:', len(linked))
linked[['Award ID', 'Action Date', 'Transaction Amount', 'Mod']].head() if len(linked) else linked

## 6. Notes for Phase 2 schema

Capture decisions while exploring:

- Preferred award PK?
- Preferred transaction PK?
- Vendor identity strategy when UEI is null?
- Which money fields map to `initial_value` / `current_value`?

Write lasting conclusions into `docs/data_dictionary.md` and `docs/sources_notes.md`.

In [ ]:
notes = {
    'preferred_award_pk_candidate': 'generated_unique_award_id',
    'vendor_id_priority': ['UEI', 'normalized_name', 'name+geo', 'fuzzy'],
    'scope': provenance.get('scope'),
}
notes